# Classifier Training — BERT 3-class Intent on Synthetic Data

Fine-tune `bert-base-uncased` cho intent classification 3 class.

**Classes:** `NUTRITION_LOOKUP` · `HEALTH_ADVICE` · `BOTH`  
**Data:** synthetic intent CSV  — 500 rows/class = 1,500 rows tổng  
**Output:** `classifier_bert/` trên Drive → `models/classifier_bert/` local  
**Eval metric:** Accuracy + F1 macro trên 20% validation split

**Format CSV**
```
text,label
"How many calories in 100g of chicken?",NUTRITION_LOOKUP
"What should I eat if I have diabetes?",HEALTH_ADVICE
"Is salmon high in omega-3 and good for heart disease?",BOTH
```


1. Tạo CSV 1,500 rows (500 × 3 class) — dùng ChatGPT/Gemini generate
2. Upload lên Drive: `MyDrive/nutrition-rag/intent_data.csv`
4. Chạy từng cell theo thứ tự

In [1]:
!pip install -q evaluate

## Setup — Mount Drive & Paths

In [2]:
import os

from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/nutrition-rag"
DATA_PATH  = f"{DRIVE_BASE}/intent_data.csv"
OUTPUT_DIR = f"{DRIVE_BASE}/classifier_bert"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.path.exists(DATA_PATH), f"Upload intent_data.csv to {DRIVE_BASE}/ first"

print(f"Data:   {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!pip install --upgrade transformers evaluate

import warnings

import numpy as np
import pandas as pd
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

warnings.filterwarnings("ignore")

MODEL_CHECKPOINT = "bert-base-uncased"
LABEL_LIST = ["NUTRITION_LOOKUP", "HEALTH_ADVICE", "BOTH"]
label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label  = {i: l for l, i in label2id.items()}

print(f"Labels: {label2id}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 96.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Labels: {'NUTRITION_LOOKUP': 0, 'HEALTH_ADVICE': 1, 'BOTH': 2}


## Data Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
assert set(df["label"].unique()) <= set(LABEL_LIST), \
    f"Unknown labels found: {set(df['label'].unique()) - set(LABEL_LIST)}"

df["label_id"] = df["label"].map(label2id)

print(f"Total: {len(df)} rows")
print(df["label"].value_counts().to_string())

# 80/20 split
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train_df, val_df = df[:split_idx], df[split_idx:]

train_ds = Dataset.from_dict({"text": train_df["text"].tolist(), "label": train_df["label_id"].tolist()})
val_ds   = Dataset.from_dict({"text": val_df["text"].tolist(),   "label": val_df["label_id"].tolist()})

print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)}")

Total: 1500 rows
label
BOTH                500
HEALTH_ADVICE       500
NUTRITION_LOOKUP    500

Train: 1200 | Val: 300


## Preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128, padding=False)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize,   batched=True, remove_columns=["text"])

print(f"Tokenized — Train: {len(train_tok)} | Val: {len(val_tok)}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenized — Train: 1200 | Val: 300


## Training

In [ ]:
import warnings

import numpy as np
import pandas as pd
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

warnings.filterwarnings("ignore")

MODEL_CHECKPOINT = "bert-base-uncased"
LABEL_LIST = ["NUTRITION_LOOKUP", "HEALTH_ADVICE", "BOTH"]
label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label  = {i: l for l, i in label2id.items()}

print(f"Labels: {label2id}")

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {
        "accuracy": accuracy,
        "f1":       f1,
    }


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    report_to="none",
)

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Training {len(train_tok)} examples | {training_args.num_train_epochs} epochs")

Labels: {'NUTRITION_LOOKUP': 0, 'HEALTH_ADVICE': 1, 'BOTH': 2}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Training 1200 examples | 5 epochs


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.372922,0.996667,0.996703
2,No log,0.017624,1.000000,1.000000
3,No log,0.006126,1.000000,1.000000
4,No log,0.004406,1.000000,1.000000
5,No log,0.003957,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=190, training_loss=0.2143487629137541, metrics={'train_runtime': 104.4977, 'train_samples_per_second': 57.418, 'train_steps_per_second': 1.818, 'total_flos': 72578191821408.0, 'train_loss': 0.2143487629137541, 'epoch': 5.0})

## Evaluation

In [ ]:
metrics = trainer.evaluate()

print(f"\nValidation results (best checkpoint):")
print(f"  Accuracy: {metrics['eval_accuracy']:.4f}")
print(f"  F1 macro: {metrics['eval_f1']:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.017577,5,1.000000,1.000000



Validation results (best checkpoint):
  Accuracy: 1.0000
  F1 macro: 1.0000


## Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved → {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved → /content/drive/MyDrive/nutrition-rag/classifier_bert


## Download — Zip & Export to Local

Giải nén vào `D:\FoodRecomendationSystem\models\classifier_bert\`

In [ ]:
import shutil
from google.colab import files

INFERENCE_FILES = [
    "config.json",
    "model.safetensors",
    "tokenizer_config.json",
    "tokenizer.json",
    "vocab.txt",
    "special_tokens_map.json",
]

EXPORT_DIR = "/content/classifier_bert_export"
os.makedirs(EXPORT_DIR, exist_ok=True)
for fname in INFERENCE_FILES:
    src = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, EXPORT_DIR)

shutil.make_archive("/content/classifier_bert", "zip", EXPORT_DIR)

zip_size = os.path.getsize("/content/classifier_bert.zip") / 1e6
print(f"Zip size: {zip_size:.1f} MB")
print("Extract to: D:\\FoodRecomendationSystem\\models\\classifier_bert\\")

files.download("/content/classifier_bert.zip")

Zip size: 405.8 MB
Extract to: D:\FoodRecomendationSystem\models\classifier_bert\


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>